# 🖼️ Google Photos AI 老照片修復與超解析度工作台

> **專為老舊照片修復打造**：整合 **GFPGAN v1.4**（五官高清重建）與 **Real-ESRGAN**（4K 無損放大與降噪）。
>
> 💡 **核心特色**：
> 1. **模糊臉孔精緻修復**：自動修復老照片中失焦、泛黃、解析度過低的人臉五官與眼睫毛細節。
> 2. **全圖 4 倍超解析度**：去除老舊相片掃描紋理與壓縮雜訊，無損提升至 4K 高畫質。
> 3. **安全無損保全**：所有修復照片輸出至獨立的 `Restored/` 資料夾，原始老照片 100% 唯讀保留。
> 4. **雙模式操作**：支援 Gradio 互動網頁（單張即時 Before/After 對比）與 Google Drive 批次整批相簿全自動修復。

## 步驟 1：檢查 GPU 執行階段

本工作台需要 GPU 進行深度學習推論。
請確認上方選單「執行階段」->「變更執行階段類型」已選擇 **T4 GPU** 或更高等級。

In [ ]:
!nvidia-smi

## 步驟 2：掛載 Google 雲端硬碟 (Google Drive)

點擊左側播放鍵 ▶，依提示授權存取 Google Drive，以讀取相簿與保存修復成果。

In [ ]:
import os, subprocess
from google.colab import drive
from pathlib import Path

def ensure_drive_connected():
    """確保 /content/drive 連線正常，若遇瞬斷自動重新掛載"""
    try:
        _ = os.listdir('/content/drive/MyDrive')
    except OSError:
        subprocess.run(['fusermount', '-uz', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
        subprocess.run(['umount', '-l', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
        drive.mount('/content/drive', force_remount=True)

try:
    _ = os.listdir('/content/drive/MyDrive')
except OSError:
    subprocess.run(['fusermount', '-uz', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
    subprocess.run(['umount', '-l', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)

drive.mount('/content/drive', force_remount=True)
DRIVE_ROOT = Path('/content/drive/MyDrive')
print('✓ Google Drive 掛載成功！')

## 步驟 3：安裝 GFPGAN、Real-ESRGAN 與 Gradio 介面套件

下載官方權重模型（約 330 MB），本步驟初次執行約需 1～2 分鐘。

In [ ]:
# 安裝基礎依賴與 HEIC 支援（鎖定 pillow<13.0 相容 Gradio）
!pip install -q gradio "pillow>=10.0,<13.0" pillow-heif opencv-python addict future lmdb pyyaml requests scikit-image scipy tqdm yapf
!pip install -q facexlib

# Python 3.13 (PEP 667) 下官方 basicsr==1.4.2 的 setup.py 使用 locals()['__version__'] 會拋出 KeyError；
# 此處動態修補官方 PyPI 原始碼後直接安裝（同時略過訓練專用的 tb-nightly 與重複建置的 setup_requires）。
import importlib.util, json, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

if importlib.util.find_spec('basicsr') is None:
    meta = json.loads(urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json', timeout=30).read())
    sdist_url = meta['urls'][0]['url']
    with tempfile.TemporaryDirectory() as tmp:
        tgz = Path(tmp) / 'basicsr-1.4.2.tar.gz'
        tgz.write_bytes(urllib.request.urlopen(sdist_url, timeout=60).read())
        with tarfile.open(tgz, 'r:gz') as tf:
            if sys.version_info >= (3, 12):
                tf.extractall(tmp, filter='data')
            else:
                tf.extractall(tmp)
        pkg_dir = Path(tmp) / 'basicsr-1.4.2'
        sp = pkg_dir / 'setup.py'
        sp_txt = sp.read_text(encoding='utf-8')
        sp_txt = sp_txt.replace("return locals()['__version__']", "return '1.4.2'")
        sp_txt = sp_txt.replace("setup_requires=['cython', 'numpy', 'torch'],", "setup_requires=[],")
        sp.write_text(sp_txt, encoding='utf-8')
        req = pkg_dir / 'requirements.txt'
        req.write_text('\n'.join(l for l in req.read_text(encoding='utf-8').splitlines() if 'tb-nightly' not in l) + '\n', encoding='utf-8')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '--no-deps', str(pkg_dir)])

# 安裝 GFPGAN 與 Real-ESRGAN（已完整備齊執行期依賴，避免拉入 tb-nightly 或重編 basicsr）
!pip install -q --no-deps gfpgan realesrgan

# 下載 GFPGAN v1.4 清晰人臉修復模型權重
!mkdir -p /content/weights
!wget -q -c https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth -P /content/weights/

# 下載 Real-ESRGAN 4x+ 通用超解析度放大模型權重
!wget -q -c https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P /content/weights/

for w_name in ('GFPGANv1.4.pth', 'RealESRGAN_x4plus.pth'):
    w_path = Path('/content/weights') / w_name
    if not w_path.is_file() or w_path.stat().st_size < 10 * 1024 * 1024:
        raise RuntimeError(f'模型權重下載失敗或檔案不完整：{w_path}')

print('✓ 核心修復套件與 AI 模型權重下載完畢！')

## 步驟 4：載入 AI 畫質修復核心引擎

In [ ]:
import os, sys
import cv2
import torch
import torchvision.transforms.functional as _tv_F
# 修補新版 torchvision (>=0.17) 移除 functional_tensor 導致 basicsr 匯入失敗的問題
sys.modules['torchvision.transforms.functional_tensor'] = _tv_F
# 修補 PyTorch 2.6+ 預設 weights_only=True 導致舊版 GFPGAN / Real-ESRGAN / facexlib 權重無法載入的問題
if not getattr(torch.load, '_gpto_weights_patched', False):
    _orig_torch_load = torch.load
    def _compat_torch_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _orig_torch_load(*args, **kwargs)
    _compat_torch_load._gpto_weights_patched = True
    torch.load = _compat_torch_load

import numpy as np
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None
import pillow_heif
pillow_heif.register_heif_opener()
from gfpgan import GFPGANer
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. 初始化 Real-ESRGAN 背景放大器 (4x)
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
bg_upsampler = RealESRGANer(
    scale=4,
    model_path='/content/weights/RealESRGAN_x4plus.pth',
    model=model,
    tile=400,
    tile_pad=10,
    pre_pad=0,
    half=True if torch.cuda.is_available() else False
)

# 2. 初始化 GFPGAN 人臉修復器
restorer = GFPGANer(
    model_path='/content/weights/GFPGANv1.4.pth',
    upscale=2,
    arch='clean',
    channel_multiplier=2,
    bg_upsampler=bg_upsampler
)

def restore_image(input_img, upscale=2):
    """執行老照片畫質修復與五官增強（含 EXIF 方向校正、灰階/RGBA 正規化與倍率同步）"""
    if input_img is None:
        return None
    upscale_val = int(upscale)
    pil_rgb = ImageOps.exif_transpose(input_img).convert('RGB')
    img_bgr = cv2.cvtColor(np.array(pil_rgb), cv2.COLOR_RGB2BGR)
    restorer.upscale = upscale_val
    if hasattr(restorer, 'face_helper') and restorer.face_helper is not None:
        restorer.face_helper.upscale_factor = upscale_val
    cropped_faces, restored_faces, restored_img = restorer.enhance(
        img_bgr,
        has_aligned=False,
        only_center_face=False,
        paste_back=True
    )
    output_rgb = cv2.cvtColor(restored_img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(output_rgb)

print(f'✓ AI 畫質修復引擎載入完成 ({device.type.upper()} 就緒)！')

## 步驟 5：啟動 Gradio 現代互動修復工作台

點擊下方產生的本機與公開連結，即可開啟視覺化修復工作台：
- **單張即時修復**：拖曳單張老照片上傳，支援放大倍率選擇與即時前後對比。
- **批次相簿修復**：直接輸入 Google Drive 資料夾路徑（支援遞迴掃描子目錄），全自動整批處理並輸出至 `Restored/`。

In [ ]:
import inspect, os, subprocess, time
from pathlib import Path
import gradio as gr
from PIL import Image, ImageOps

if 'ensure_drive_connected' not in globals():
    def ensure_drive_connected():
        try:
            _ = os.listdir('/content/drive/MyDrive')
        except OSError:
            from google.colab import drive
            subprocess.run(['fusermount', '-uz', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
            subprocess.run(['umount', '-l', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
            drive.mount('/content/drive', force_remount=True)

def batch_restore(folder_path, upscale_val, progress=gr.Progress()):
    ensure_drive_connected()
    target_dir = Path(folder_path.strip())
    if not target_dir.exists():
        return f'❌ 找不到目錄：{folder_path}'

    out_dir = target_dir / 'Restored'
    out_dir.mkdir(parents=True, exist_ok=True)

    valid_exts = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.heic', '.heif'}
    img_files = [
        p for p in target_dir.rglob('*')
        if p.is_file()
        and p.suffix.lower() in valid_exts
        and not p.name.startswith('.')
        and 'Restored' not in p.parts
        and 'Duplicates' not in p.parts
    ]

    if not img_files:
        return f'⚠️ 在 {folder_path} 內未找到有效相片檔案。'

    success_count = 0
    total = len(img_files)
    for idx, f in enumerate(img_files):
        progress((idx + 1) / total, desc=f'修復中 [{idx+1}/{total}]: {f.name}')
        rel_path = f.relative_to(target_dir)
        save_name = f'restored_{f.stem}.jpg' if f.suffix.lower() in ('.heic', '.heif') else f'restored_{f.name}'
        save_path = out_dir / rel_path.parent / save_name
        for _attempt in range(2):
            try:
                save_path.parent.mkdir(parents=True, exist_ok=True)
                if save_path.exists():
                    success_count += 1
                    break
                with Image.open(f) as im:
                    img = ImageOps.exif_transpose(im).convert('RGB')
                res = restore_image(img, upscale=int(upscale_val))
                tmp_save = save_path.parent / f'.gpto_tmp_{save_name}'
                res.save(tmp_save, quality=95)
                tmp_save.replace(save_path)
                success_count += 1
                break
            except OSError as exc:
                if _attempt == 0 and (getattr(exc, 'errno', None) == 107 or 'Transport endpoint is not connected' in str(exc)):
                    ensure_drive_connected()
                    time.sleep(1.0)
                    continue
                print(f'處理失敗 {f.name}: {exc}')
                break
            except Exception as e:
                print(f'處理失敗 {f.name}: {e}')
                break

    return f'🎉 批次修復完成！共處理 {success_count}/{total} 張相片，已保留目錄結構保存至：\n{out_dir}'

theme = gr.themes.Soft()
blocks_kwargs = {'title': 'Google Photos AI 老照片修復工作台'}
launch_kwargs = {'share': True, 'debug': False}
if 'theme' in inspect.signature(gr.Blocks.__init__).parameters:
    blocks_kwargs['theme'] = theme
elif 'theme' in inspect.signature(gr.Blocks.launch).parameters:
    launch_kwargs['theme'] = theme

with gr.Blocks(**blocks_kwargs) as demo:
    gr.Markdown('# 🖼️ Google Photos AI 老照片修復與超解析度工作台')
    gr.Markdown('整合 GFPGAN v1.4 人臉細節重建與 Real-ESRGAN 4x+ 高畫質放大技術。')

    with gr.Tab('✨ 單張即時修復'):
        with gr.Row():
            with gr.Column():
                in_image = gr.Image(type='pil', label='上傳欲修復的原始相片')
                upscale_factor = gr.Radio([2, 4], value=2, label='超解析度放大倍率')
                btn_restore = gr.Button('🚀 立即執行 AI 修復', variant='primary')
            with gr.Column():
                out_image = gr.Image(type='pil', label='修復後的高畫質相片')
        btn_restore.click(restore_image, inputs=[in_image, upscale_factor], outputs=[out_image])

    with gr.Tab('📁 批次相簿全自動修復'):
        gr.Markdown('### 指定 Google Drive 上的資料夾進行全自動批次修復')
        default_path = str(DRIVE_ROOT / 'Organized_Photos')
        folder_input = gr.Textbox(value=default_path, label='目標相簿資料夾絕對路徑')
        batch_upscale = gr.Radio([2, 4], value=2, label='放大倍率')
        btn_batch = gr.Button('⚡ 開始全自動批次修復', variant='primary')
        batch_result = gr.Textbox(label='執行進度與結果回報', interactive=False)
        btn_batch.click(batch_restore, inputs=[folder_input, batch_upscale], outputs=[batch_result])

demo.launch(**launch_kwargs)